In [30]:
from google.colab import drive

drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [41]:
from google.colab import files
import os

print('Please upload Brain Tumor YOLO dataset.zip:')
uploaded = files.upload()

for filename in uploaded.keys():
    if filename.endswith('.zip'):
        os.rename(filename, 'data.zip')
        print(f'Successfully uploaded and renamed {filename} to data.zip')
    else:
        print(f'Uploaded {filename}, but it is not a zip file.')

Please upload Brain Tumor YOLO dataset.zip:


Saving Brain Tumor YOLO dataset.zip to Brain Tumor YOLO dataset.zip
Successfully uploaded and renamed Brain Tumor YOLO dataset.zip to data.zip


In [43]:
### 2. Prepare data ###
import os
import shutil

if os.path.exists('/content/data.zip'):
    # Unzip to a temporary directory
    temp_dir = '/content/temp_extract'
    if os.path.exists(temp_dir): shutil.rmtree(temp_dir)
    !unzip -q '/content/data.zip' -d {temp_dir}

    # Find the top-level directory/files in the zip
    top_level = [f for f in os.listdir(temp_dir) if not f.startswith('__MACOSX')]

    if top_level:
        # If it's a single folder, move its contents; if it's multiple files, move the whole temp dir
        source = os.path.join(temp_dir, top_level[0])
        target = '/content/brain_tumor_yolo_dataset'

        if os.path.exists(target): shutil.rmtree(target)

        if os.path.isdir(source) and len(top_level) == 1:
            shutil.move(source, target)
        else:
            shutil.move(temp_dir, target)

        print(f'Dataset prepared successfully in: {target}')
        !rm -rf {temp_dir}
    else:
        print('Error: Zip file is empty or invalid.')
else:
    print('Error: data.zip not found. Please upload it in the previous cell.')

Dataset prepared successfully in: /content/brain_tumor_yolo_dataset


In [32]:
### 3. Install packages ###

!git clone https://github.com/autogyro/yolo-V8.git
!cd yolo-V8/ && pip install ultralytics

fatal: destination path 'yolo-V8' already exists and is not an empty directory.


In [ ]:
### 4. Train model ###
import os
from ultralytics import YOLO

# The dataset was extracted to /content/brain_tumor_yolo_dataset
# Let's locate the data.yaml inside it
config_path = '/content/brain_tumor_yolo_dataset/data.yaml'

if os.path.exists(config_path):
    !yolo task=detect mode=train model=yolov8n.pt data={config_path} epochs=50 imgsz=640 plots=True
else:
    print(f'Error: {config_path} not found. Checking directory contents...')
    print(os.listdir('/content/brain_tumor_yolo_dataset'))

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/brain_tumor_yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-7, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=Tru

In [34]:
# ------------------ Run YOLOv8 Validation ------------------
import os

# Update path to the newly trained weights and the config file
model_path = '/content/runs/detect/train/weights/best.pt'
config_path = '/content/brain_tumor_yolo_dataset/data.yaml'

if os.path.exists(model_path):
    !yolo task=detect mode=val model={model_path} data={config_path} save_json=True
else:
    print('Training must complete successfully before running validation.')

Traceback (most recent call last):
  File "/usr/local/bin/yolo", line 8, in <module>
    sys.exit(entrypoint())
             ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/__init__.py", line 976, in entrypoint
    model = YOLO(model, task=task)
            ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/models/yolo/model.py", line 76, in __init__
    super().__init__(model=model, task=task, verbose=verbose)
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/model.py", line 144, in __init__
    self._load(model, task=task)
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/model.py", line 283, in _load
    self.model, self.ckpt = load_checkpoint(weights)
                            ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/nn/tasks.py", line 1515, in load_checkpoint
    ckpt, weight = torch_safe_load(weight)  # load ckpt
                   ^^^^^^^^^

In [35]:
from IPython.display import Image, display
import glob

!yolo predict model=/content/runs/detect/train/weights/best.pt \
              source=/content/test/images/y701_jpg.rf.81a4472f77fdc1f31537342ceca340c9.jpg \
              project=/content/runs/detect \
              name=predict \
              exist_ok=True


# Find the path of the predicted image (YOLOv8 saves output in 'runs/detect/predict' by default)
predicted_images = glob.glob('/content/runs/detect/predict/*.jpg')  # or .png if needed

# Display the first predicted image
if predicted_images:
    display(Image(filename=predicted_images[0]))
else:
    print("No predicted images found.")


Traceback (most recent call last):
  File "/usr/local/bin/yolo", line 8, in <module>
    sys.exit(entrypoint())
             ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/__init__.py", line 976, in entrypoint
    model = YOLO(model, task=task)
            ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/models/yolo/model.py", line 76, in __init__
    super().__init__(model=model, task=task, verbose=verbose)
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/model.py", line 144, in __init__
    self._load(model, task=task)
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/model.py", line 283, in _load
    self.model, self.ckpt = load_checkpoint(weights)
                            ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/nn/tasks.py", line 1515, in load_checkpoint
    ckpt, weight = torch_safe_load(weight)  # load ckpt
                   ^^^^^^^^^

In [37]:
from ultralytics import YOLO
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Paths
test_folder = '/content/test/images'
output_folder = '/content/runs/detect'
output_name = 'predictions'
output_path = os.path.join(output_folder, output_name)

# Ensure output directory exists
os.makedirs(output_path, exist_ok=True)

# Load model
model = YOLO('/content/runs/detect/train/weights/best.pt')

# Select up to 16 test images
test_images = [os.path.join(test_folder, img) for img in os.listdir(test_folder) if img.endswith(('.jpg', '.png'))][:16]

# Run predictions and save
results = []
for img_path in test_images:
    result = model.predict(source=img_path, save=True, save_txt=True, project=output_folder, name=output_name, exist_ok=True)
    results.append(result)

# Gather predicted image paths
predicted_images = [os.path.join(output_path, os.path.basename(img_path)) for img_path in test_images]

# Display images in 4x4 grid
fig, axes = plt.subplots(4, 4, figsize=(16, 16))
axes = axes.flatten()

for i, img_path in enumerate(predicted_images):
    if os.path.exists(img_path):
        img = mpimg.imread(img_path)
        axes[i].imshow(img)
        axes[i].set_title(os.path.basename(img_path), fontsize=8)
        axes[i].axis('off')
    else:
        axes[i].axis('off')
        axes[i].set_title("Image not found", fontsize=8)

plt.tight_layout()
plt.show()


FileNotFoundError: [Errno 2] No such file or directory: '/content/runs/detect/train/weights/best.pt'

In [ ]:
### 5. Download results ###

from google.colab import files


!zip -r /content/runs.zip /content/runs

files.download('/content/runs.zip')